# Phase 2 — Geological & domain understanding

Build intuition before modeling: **TVT vs MD**, **GR** (known vs eval zone), **horizontal ↔ typewell** correlation, **XY** footprint, **dTVT/dMD**.

**Prerequisites:** `data/raw/` populated (`python scripts/download_data.py`) and Phase 0–1 notebook assumptions verified.

**Run from repo root** (or any folder under the repo) so `import src` resolves — same pattern as notebook `01_phase0_phase1_data_and_schema.ipynb`.

---

In [ ]:
## Setup — imports, paths, and well bundles

Run this cell once per session. It builds the **`well_bundle_train`** / **`well_bundle_test`** objects you will slice and plot in the cells below.


In [ ]:
import sys
from pathlib import Path

# --- Repo root (same logic as notebook 01) ---
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for p in ROOT.parents:
        if (p / "src").is_dir():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.io.loaders import discover_wells, load_all_wells, load_sample_submission
from src.utils.paths import get_artifacts_dir, get_data_dir, get_sample_submission_path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

# --- Paths ---
train_dir = get_data_dir("train")
test_dir = get_data_dir("test")
sample_path = get_sample_submission_path()

for label, p in [("train", train_dir), ("test", test_dir), ("sample_submission", sample_path)]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {label}: {p}. Run `python scripts/download_data.py` or complete notebook 01."
        )

ARTIFACTS_FIGURES = get_artifacts_dir("figures")
ARTIFACTS_FIGURES.mkdir(parents=True, exist_ok=True)

# --- Well lists & submission template ---
train_wells = discover_wells(train_dir)
test_wells = discover_wells(test_dir)
sub = load_sample_submission(sample_path)

# --- Primary working objects: nested dicts of DataFrames ---
# well_bundle_train[well_id]["horizontal"] — lateral CSV (MD, XYZ, GR, TVT_input, TVT, …)
# well_bundle_train[well_id]["typewell"]   — vertical reference (TVT, GR, Geology)
well_bundle_train = load_all_wells(train_dir)
well_bundle_test = load_all_wells(test_dir)

# Scratch aliases on one example well (change EXAMPLE_WELL anytime)
EXAMPLE_WELL = sorted(train_wells)[0]
df_horiz_ex = well_bundle_train[EXAMPLE_WELL]["horizontal"]
df_typewell_ex = well_bundle_train[EXAMPLE_WELL]["typewell"]

print("ROOT:", ROOT)
print("train_dir:", train_dir)
print("test_dir:", test_dir)
print("ARTIFACTS_FIGURES:", ARTIFACTS_FIGURES)
print(f"Train wells: {len(train_wells)} | loaded bundles: {len(well_bundle_train)}")
print(f"Test wells: {len(test_wells)} | loaded bundles: {len(well_bundle_test)}")
print(f"sample_submission (eval rows total): {len(sub)}")
print(f"Example `{EXAMPLE_WELL}`: horizontal {df_horiz_ex.shape}, typewell {df_typewell_ex.shape}")


### Reference — what each object is

| Name | Type | Use in Phase 2 |
|------|------|----------------|
| `ROOT` | `Path` | Repo root; saves figures under `ROOT/artifacts/…` if needed. |
| `train_dir` / `test_dir` | `Path` | Raw Kaggle folders (`data/raw/train`, `data/raw/test`). |
| `sample_path` | `Path` | `sample_submission.csv`. |
| `ARTIFACTS_FIGURES` | `Path` | Save plots here (`eda__…__YYYY-MM-DD.png` per `CURSOR_RULES.md`). |
| `train_wells` / `test_wells` | `list[str]` | Sorted well ids from `*__horizontal_well.csv` filenames. |
| `sub` | `DataFrame` | Columns `id`, `tvt`; helpers `wellname`, `row_idx` from `load_sample_submission`. |
| **`well_bundle_train`** | `dict[str, dict]` | **`well_bundle_train[w]["horizontal"]`** → lateral `DataFrame`; **`["typewell"]`** → typewell `DataFrame`. |
| **`well_bundle_test`** | same shape | Test split (no `TVT`, no train-only formation cols). |
| `df_horiz_ex` / `df_typewell_ex` | `DataFrame` | Views of one example well — swap `EXAMPLE_WELL` and re-run setup if you like. |

**Heavy note:** `load_all_wells` loads **every** train well into RAM (~773 laterals + typewells). For lighter probes use `load_horizontal_well(train_dir / f"{w}__horizontal_well.csv")` on a subset of `w` only.


In [ ]:
# Peek at the lateral frame (example well)
df_horiz_ex.head()


In [ ]:
# Peek at the typewell frame (same well)
df_typewell_ex.head()
